# Logistic Regression

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

FEATURES = [
    "pH", "Temperature", "Ammonia", "Nitrite", "Nitrate", "DissolvedO2",
    "Min pH", "Max pH", "Min Temp (C)", "Max Temp (C)",
    "Max Safe Ammonia (ppm)", "Max Safe Nitrite (ppm)",
    "Max Safe Nitrate (ppm)", "Min Dissolved O2 (mg/L)",
]
PARAMS = ["pH", "Temperature", "Ammonia", "Nitrite", "Nitrate", "DissolvedO2"]
LABELS = ["excellent", "good", "watch", "critical"]

df = pd.read_csv(next(Path("/kaggle/input").rglob("water_quality_training_samples.csv")))
X = df[FEATURES]
enc = LabelEncoder().fit(LABELS)
y = enc.transform(df["Quality"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="Quality", order=LABELS, ax=axes[0])
axes[0].set_title("Quality counts")
sns.countplot(data=df, x="Kind", ax=axes[1])
axes[1].set_title("Fish vs Plant")
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), PARAMS):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(col)
plt.suptitle("Parameter distributions")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 6))
sns.heatmap(df[PARAMS].corr(), annot=True, fmt=".2f", cmap="YlGnBu", vmin=-1, vmax=1)
plt.title("Correlation heatmap")
plt.show()

plt.figure(figsize=(9, 4))
sns.heatmap(df.groupby("Quality")[PARAMS].mean().reindex(LABELS), annot=True, fmt=".2f", cmap="YlOrRd")
plt.title("Mean values by quality")
plt.show()

In [ ]:
model = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)
model.fit(X_train_s, y_train)
pred = model.predict(X_test_s)

print("Accuracy", round(accuracy_score(y_test, pred) * 100, 2))
print("F1", round(f1_score(y_test, pred, average="macro") * 100, 2))
print(classification_report(y_test, pred, target_names=LABELS))

plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt="d", cmap="Blues",
            xticklabels=LABELS, yticklabels=LABELS)
plt.title("Confusion matrix")
plt.show()

pd.Series(np.abs(model.coef_).mean(axis=0), index=FEATURES).sort_values().plot(kind="barh", figsize=(8, 6))
plt.title("Feature importance")
plt.show()

joblib.dump(
    {"model": model, "scaler": scaler, "label_encoder": enc, "features": FEATURES, "labels": LABELS},
    "/kaggle/working/logistic_regression_water.pkl",
)